# 03 - Data splits and YOLO preparation
1. Builds **leakage-aware** train/val/test splits by **(site, day)**: frames of one day never cross splits, and the nine CNRPark-EXT cameras share days so a day goes to one split for the whole site. Climates are checked to appear in val and test.
2. Tags non-working / non-representative days (kept, not removed).
3. Saves the split manifest and its hash. **The test split is frozen from here on: no tuning on it.**
4. Exports the YOLO dataset (single class `car`). Pixels outside each image's `annotationsRectangle` are grayed out because cars there are not labelled. Only the **train** split is subsampled (`SUBSET_FRACTION`); val/test are always complete.

**Inputs:** MetaPKLot dataset (required); notebook 02 output `annotations_v2/` (optional, saves ~1.5 min).
**Outputs:** `/kaggle/working/artifacts/splits/` (manifest, info, status) and `/kaggle/working/yolo_f<fraction>/` (images, labels, `data.yaml`).
CPU is enough. Then **Save Version** so notebook 04 (training) can attach the output.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Step 2: configuration. The dataset folder is detected automatically.
DATASET_HINT = ""       # only needed if several datasets are attached: part of the right path, e.g. "metapklot"
DATA_ROOT = None        # or set the path yourself to skip detection

from src.data.discover import describe, find_dataset_root
if DATA_ROOT is None:
    DATA_ROOT = str(find_dataset_root("/kaggle/input", DATASET_HINT))
info = describe(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("files:", info["files"], "| top-level entries:", info["n_children"], info["children"])
# Settings (defaults come from configs/dataset.yaml; override here if needed)
from src.common.config import load_yaml, config_hash
CFG = load_yaml("configs/dataset.yaml")
SUBSET_FRACTION = CFG["dataset"]["subset_fraction"]   # train only; 1.0 for the full-data run
SPLIT = CFG["split"]
OUT = "/kaggle/working/artifacts/splits"
YOLO_OUT = "/kaggle/working/yolo_f{}".format(SUBSET_FRACTION)
print("subset_fraction:", SUBSET_FRACTION, "| split cfg:", SPLIT, "| yolo out:", YOLO_OUT)

In [ ]:
# Cell 3: load images, cars and spots (from notebook 02 output if attached, else from the dataset)
import csv, glob, gzip, json, time
from pathlib import Path
from src.data import metapklot as mp
root = Path(DATA_ROOT)
t0 = time.time()
attached = sorted(glob.glob("/kaggle/input/**/annotations_v2/images.csv", recursive=True))
if attached:
    base = os.path.dirname(attached[0])
    print("using notebook 02 output:", base)
    images = []
    with open(base + "/images.csv") as f:
        for r in csv.DictReader(f):
            r["image_id"] = int(r["image_id"]); r["width"] = int(r["width"]); r["height"] = int(r["height"])
            r["rect"] = json.loads(r["rect"]) if r["rect"] else None
            images.append(r)
    boxes = {}
    with gzip.open(base + "/cars.csv.gz", "rt") as f:
        for r in csv.DictReader(f):
            boxes.setdefault(int(r["image_id"]), []).append(json.loads(r["bbox"]))
    spots = []
    with gzip.open(base + "/spots.csv.gz", "rt") as f:
        for r in csv.DictReader(f):
            spots.append((int(r["image_id"]), int(r["category_id"])))
else:
    print("notebook 02 output not attached: loading annotations from the dataset")
    S = mp.load_set(root, CFG["dataset"]["annotation_variant"], "spots", CFG["dataset"]["sources"])
    C = mp.load_set(root, CFG["dataset"]["annotation_variant"], "cars", CFG["dataset"]["sources"])
    images = S["images"]
    boxes = {}
    for a in C["annotations"]:
        boxes.setdefault(a["image_id"], []).append(a["bbox"])
    spots = [(a["image_id"], a["category_id"]) for a in S["annotations"]]
images_by_id = {i["image_id"]: i for i in images}
print("images:", len(images), "| images with cars:", len(boxes), "| car boxes:", sum(map(len, boxes.values())),
      "| spots:", len(spots), "| loaded in", round(time.time() - t0, 1), "s")

In [ ]:
# Cell 4: day flags, split assignment, leakage check, manifest
from src.data import splits as sp
day_flags = sp.parse_day_lists(mp.read_day_lists(root))
assign, info = sp.assign_splits(images, SPLIT["ratios"], SPLIT["seed"], holdout_cameras=SPLIT["holdout_cameras"],
                                min_days_for_coverage=SPLIT["min_days_for_coverage"])
keep = sp.subsample_train(images, assign, SUBSET_FRACTION)
rows = sp.manifest_rows(images, assign, keep, day_flags)
leaks = sp.check_no_leakage(rows)
assert not leaks, leaks[:5]
mhash = sp.manifest_hash(rows)
print("seed used:", info["seed_used"], "| tries:", info["tries"], "| climate coverage ok:", info["climate_coverage_ok"])
print("manifest hash:", mhash, "| leakage violations:", len(leaks))
for site, d in info["days_per_split"].items():
    print(" ", site, {k: len(v) for k, v in d.items()})
print(json.dumps(sp.split_summary(rows), indent=1))

In [ ]:
# Cell 5: occupancy balance per split (slot-level labels)
from collections import Counter
occ = Counter(); tot = Counter()
for image_id, cat in spots:
    s = assign[image_id]; tot[s] += 1; occ[s] += int(cat == 1)
for s in sorted(tot):
    print("{:13s} slots: {:>8d}  occupied: {:.1%}".format(s, tot[s], occ[s] / tot[s]))

In [ ]:
# Cell 6: save the manifest (frozen) and split info
from src.common.status import new_run_id, write_status
os.makedirs(OUT, exist_ok=True)
with open(OUT + "/split_manifest.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
info.update({"manifest_hash": mhash, "subset_fraction": SUBSET_FRACTION, "config_hash": config_hash(CFG),
             "summary": sp.split_summary(rows), "flagged_days": {"{}|{}".format(*k): v for k, v in day_flags.items()}})
with open(OUT + "/split_info.json", "w") as f:
    json.dump(info, f, indent=2)
RUN_ID = new_run_id("split")
write_status(OUT + "/status.json", "03_splits", RUN_ID, "RUNNING", config_hash(CFG), "manifest saved; export pending")
print("saved", OUT, sorted(os.listdir(OUT)))

In [ ]:
# Cell 7: export the YOLO dataset (resumable: re-running skips files already written). ~5-15 min.
from src.data import yolo_export as ye
t0 = time.time()
counts = ye.export_dataset(root, YOLO_OUT, rows, images_by_id, boxes, workers=CFG["export"]["workers"])
yaml_path = ye.write_data_yaml(YOLO_OUT)
print(counts, "| data.yaml:", yaml_path, "| took", round(time.time() - t0, 1), "s")
for s in ("train", "val", "test"):
    p = YOLO_OUT + "/images/" + s
    print(s, "images:", len(os.listdir(p)) if os.path.isdir(p) else 0)
print("disk used by export: {:.2f} GB".format(sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(YOLO_OUT) for f in fs) / 1e9))

In [ ]:
# Cell 8: visual check - draw the exported YOLO labels back onto two exported train images
from PIL import Image, ImageDraw
from IPython.display import display
for name in sorted(os.listdir(YOLO_OUT + "/images/train"))[::max(1, len(os.listdir(YOLO_OUT + "/images/train")) // 2)][:2]:
    im = Image.open(YOLO_OUT + "/images/train/" + name).convert("RGB"); d = ImageDraw.Draw(im)
    lbl = YOLO_OUT + "/labels/train/" + os.path.splitext(name)[0] + ".txt"
    for line in open(lbl).read().strip().splitlines():
        _, cx, cy, w, h = map(float, line.split())
        W, H = im.size
        d.rectangle([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H], outline=(0, 220, 220), width=2)
    print(name); display(im.resize((im.width // 2, im.height // 2)))

In [ ]:
# Cell 9: finish
write_status(OUT + "/status.json", "03_splits", RUN_ID, "COMPLETE", config_hash(CFG),
             "subset_fraction={} manifest={}".format(SUBSET_FRACTION, mhash), extra={"export": counts})
print("DONE. Save Version, then attach this notebook's output to notebook 04.")

## What to send back
Paste the output of cells 3, 4, 5 and 7, and say whether the boxes in cell 8 sit on the cars (gray band = unlabelled area outside the annotation rectangle). Then **Save Version** (Save & Run All or Quick Save with outputs) so `yolo_f0.1/` and `artifacts/splits/` become an attachable dataset for notebook 04.